In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import importlib.metadata

try:
    importlib.metadata.version('venn')
    importlib.metadata.version('pyyaml')
    importlib.metadata.version('scikit-learn')
except importlib.metadata.PackageNotFoundError:
    print("Installing missing packages...")
    ! python -m pip install venn pyyaml scikit-learn

# Get max curated data

In [ ]:
import requests
from pathlib import Path
from yaml import safe_load
import numpy as np
import re

DATA_DIR = Path("data/max-curated")
DATA_DIR.mkdir(exist_ok=True, parents=True)

activities = ["Ki", "IC50"]
data_url = "https://raw.githubusercontent.com/rinikerlab/overlapping_assays/refs/heads/main/datasets/{dataset}_datasets.yaml"


command_str = (
    "capricho get -tids {targets} -chiral -duchi -reqdoc "
    "-cure --max-assay-size 100 --min-assay-size 20 --min-assay-overlap 5 -smr -calc -c 9 -idcols assay_type,assay_cell_type "
    "-biotype {activity} -o {output}"
)

## Ki

In [ ]:
result = requests.get(data_url.format(dataset=activities[0]))
result.raise_for_status()

datasets = safe_load(result.content)

ki_targets = []
keys = datasets["sources"].keys()
for key in keys:
    ki_targets.append(datasets["sources"][key]["description"])

ki_targets = np.unique(ki_targets).tolist()
ki_targets = [re.compile("(CHEMBL\d+)").search(t).group(1) for t in ki_targets]

ki_output = DATA_DIR / "curated-Ki-NoAssayOverlap.csv"
command = command_str.format(targets=",".join(ki_targets), activity=activities[0], output=str(ki_output))

if not ki_output.exists():
    ! {command}
else:
    print(f"Data already exists: {ki_output}")
    print("Delete this file to re-fetch with updated parameters.")

## IC50

In [ ]:
result = requests.get(data_url.format(dataset=activities[1]))
result.raise_for_status()

datasets = safe_load(result.content)

ic50_targets = []
keys = datasets["sources"].keys()
for key in keys:
    ic50_targets.append(datasets["sources"][key]["description"])

ic50_targets = np.unique(ic50_targets).tolist()
ic50_targets = [re.compile("(CHEMBL\d+)").search(t).group(1) for t in ic50_targets]

ic50_output = DATA_DIR / "curated-IC50-NoAssayOverlap.csv"
command = command_str.format(targets=",".join(ic50_targets), activity=activities[1], output=str(ic50_output))

if not ic50_output.exists():
    ! {command}
else:
    print(f"Data already exists: {ic50_output}")
    print("Delete this file to re-fetch with updated parameters.")

In [ ]:
from venn import venn

venn_dict = {"IC50": set(ic50_targets), "Ki": set(ki_targets)}
venn(venn_dict, figsize=(4, 4))

plt.show()

# IC50 overlap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from Capricho.analysis import (
    DroppingComment,
    ProcessingComment,
    explode_assay_comparability,
    get_all_comments,
    plot_multi_panel_comparability,
    plot_subset,
)

In [ ]:
df = pd.read_csv(ic50_output, engine="pyarrow")
subset = df.query('processed_smiles.str.contains("|", regex=False)').assign(repeat=lambda x: range(len(x)))
subset.shape

In [ ]:
df.data_processing_comment.apply(
    lambda x: x.split("|") if isinstance(x, str) else []
).explode().value_counts()

In [ ]:
all_comments = get_all_comments()
exploded_subset = explode_assay_comparability(subset)

In [ ]:
fig, axs = plot_multi_panel_comparability(
    exploded_subset, all_comments, title="IC50 Comparability Across Flagged Data", figsize=(22, 8), ncols=5
)
fig.savefig("ic50_comparability_allflags.png", dpi=300, bbox_inches="tight", transparent=False)

In [ ]:
drop_flags_rm = [
    DroppingComment.DATA_VALIDITY_COMMENT.value,
    DroppingComment.POTENTIAL_DUPLICATE.value,
]

drop_flags_rm = "|".join(drop_flags_rm)

fig, ax = plot_subset(
    exploded_subset.query("(~dropping_comment.str.contains(@drop_flags_rm, regex=True))").query(
        "pchembl_value_x != pchembl_value_y"
    ),
    title="Unprocessed IC50 Data",
    figsize=(5, 4.6),
)

fig.savefig("ic50_unprocessed_comparability.png", dpi=300, bbox_inches="tight", transparent=False)

In [ ]:
drop_flags_rm = [
    DroppingComment.DATA_VALIDITY_COMMENT.value,
    DroppingComment.POTENTIAL_DUPLICATE.value,
    DroppingComment.ASSAY_SIZE_TOO_SMALL.value,
    DroppingComment.MUTATION_KEYWORD.value,
    DroppingComment.UNDEFINED_STEREOCHEMISTRY.value,
    DroppingComment.UNIT_ANNOTATION_ERROR.value,
]
rm_process_obs = ProcessingComment.PCHEMBL_DUPLICATION_ACROSS_DOCUMENTS.value

drop_flags_rm = "|".join(drop_flags_rm)

subset_ki_clean = exploded_subset.query(
    "(~dropping_comment.str.contains(@drop_flags_rm, regex=True)) "
    "& ~processing_comment.str.contains(@rm_process_obs, regex=True)"
    "& pchembl_value_x != pchembl_value_y"
)
fig, ax = plot_subset(subset_ki_clean, title="Cleaned IC50 Data", figsize=(5, 4.6))
fig.savefig("ic50_clean_comparability.png", dpi=300, bbox_inches="tight", transparent=False)

# Ki overlap

In [ ]:
df_ki = pd.read_csv(ki_output, engine="pyarrow")
subset_ki = df_ki.query('processed_smiles.str.contains("|", regex=False)').assign(
    repeat=lambda x: range(len(x))
)
subset_ki.shape

In [ ]:
all_comments = get_all_comments()
exploded_subset_ki = explode_assay_comparability(subset_ki)

In [ ]:
fig, axs = plot_multi_panel_comparability(
    exploded_subset_ki, all_comments, title="Ki Comparability Across Flagged Data", figsize=(22, 8), ncols=5
)
fig.savefig("ki_flagged_comparability.png", dpi=300, bbox_inches="tight", transparent=False)

In [ ]:
drop_flags_rm = [
    DroppingComment.DATA_VALIDITY_COMMENT.value,
    DroppingComment.POTENTIAL_DUPLICATE.value,
]

drop_flags_rm = "|".join(drop_flags_rm)

fig, ax = plot_subset(
    exploded_subset_ki.query("(~dropping_comment.str.contains(@drop_flags_rm, regex=True))").query(
        "pchembl_value_x != pchembl_value_y"
    ),
    title="Unprocessed Ki Data",
    figsize=(5, 4.6),
)

fig.savefig("ki_unprocessed_comparability.png", dpi=300, bbox_inches="tight", transparent=False)

In [ ]:
drop_flags_rm = [
    DroppingComment.DATA_VALIDITY_COMMENT.value,
    DroppingComment.POTENTIAL_DUPLICATE.value,
    DroppingComment.ASSAY_SIZE_TOO_LARGE.value,
    DroppingComment.ASSAY_SIZE_TOO_SMALL.value,
]
rm_process_obs = ProcessingComment.PCHEMBL_DUPLICATION_ACROSS_DOCUMENTS.value


drop_flags_rm = "|".join(drop_flags_rm)


subset_ki_clean = exploded_subset_ki.query(
    "(~dropping_comment.str.contains(@drop_flags_rm, regex=True)) "
    "& ~processing_comment.str.contains(@rm_process_obs, regex=True)"
    "& pchembl_value_x != pchembl_value_y"
).query("pchembl_value_x != pchembl_value_y")
fig, ax = plot_subset(subset_ki_clean, title="Cleaned Ki Data", figsize=(5, 4.6))

fig.savefig("ki_clean_comparability.png", dpi=300, bbox_inches="tight", transparent=False)

In [ ]:
potential_duplicates = subset_ki_clean = exploded_subset_ki.query(
    "(~dropping_comment.str.contains(@drop_flags_rm, regex=True)) "
    "& ~processing_comment.str.contains(@rm_process_obs, regex=True)"
).query("pchembl_value_x == pchembl_value_y")